# Đề tài: Khai thác mẫu, phân tích cụm và kiểm định thống kê cho mô hình gom cụm văn bản báo chí tiếng Việt

## Bối cảnh
Sau Notebook 4, mô hình **Semantic Embedding + UMAP(50) + K-Means(k=6)** đã được xác định là cấu hình tốt nhất hiện tại cho bài toán gom cụm văn bản báo chí tiếng Việt. Kết quả cho thấy mô hình semantic vượt trội rõ rệt so với baseline lexical trên cả các chỉ số có giám sát và không giám sát. Tuy nhiên, để bám sát đầy đủ yêu cầu của đồ án, vẫn cần tiếp tục thực hiện các bước khai thác mẫu, phân tích cụm sâu hơn và kiểm định thống kê nhằm giải thích rõ bản chất từng cụm và chứng minh độ ổn định của mô hình.

## Mục tiêu
Notebook này tập trung vào ba nhóm nhiệm vụ chính:
- sử dụng kết quả từ mô hình semantic tốt nhất để gắn nhãn cụm vào dữ liệu,
- thực hiện khai thác mẫu và phân tích cụm, bao gồm mô tả đặc trưng của từng cụm, từ khóa nổi bật và văn bản đại diện,
- thực hiện kiểm định thống kê để so sánh mô hình semantic với baseline lexical và đánh giá xem sự cải thiện có mang ý nghĩa thống kê hay không.

## Ý nghĩa học thuật
Notebook này đóng vai trò hoàn thiện phần phân tích sau mô hình. Nếu các notebook trước tập trung vào xây dựng pipeline và tối ưu mô hình gom cụm, thì notebook này giúp trả lời các câu hỏi quan trọng hơn ở mức diễn giải: cụm nào tương ứng với chuyên mục nào, cụm nào còn bị trộn, những mẫu từ nào xuất hiện nổi bật trong từng cụm và liệu mô hình tốt nhất có thực sự ổn định hơn baseline hay không. Đây là phần giúp kết nối giữa kết quả định lượng và giá trị học thuật của đồ án.

## Đầu ra mong đợi
Sau notebook này, đầu ra mong đợi gồm:
- dataframe đã được gắn nhãn cụm semantic tốt nhất,
- các transaction phục vụ khai thác mẫu theo từng cụm,
- các mẫu phổ biến tối đại hoặc từ khóa đặc trưng cho mỗi cụm,
- danh sách văn bản đại diện cho từng cụm,
- bảng kết quả kiểm định thống kê giữa baseline lexical và mô hình semantic,
- và kết luận cuối cùng đủ cơ sở để chốt mô hình tốt nhất cho đồ án.

In [1]:
# Cell 2: import thư viện và khai báo dữ liệu đầu vào cho Notebook 5
# Cell này chuẩn bị môi trường làm việc để phân tích cụm, khai thác mẫu và kiểm định thống kê
# dựa trên mô hình semantic tốt nhất đã chọn ở Notebook 4.

import pandas as pd
import numpy as np
import re
import time
from pathlib import Path
from collections import Counter

#INPUT_PATH = Path("D:/BAI TEST/data/processed_data/bai_test_do_an_preprocessed.csv")
INPUT_PATH = Path("bai_test_do_an_preprocessed.csv")
OUTPUT_DIR = Path("processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("File đầu vào:", INPUT_PATH)
print("Tồn tại:", INPUT_PATH.exists())

print("\nThư mục đầu ra:", OUTPUT_DIR)

File đầu vào: bai_test_do_an_preprocessed.csv
Tồn tại: True

Thư mục đầu ra: processed_data


In [2]:
# Cell 3: đọc file tiền xử lý và gắn lại kết quả cụm semantic tốt nhất
# Cell này chuẩn bị dataframe nền cho toàn bộ phần khai thác mẫu và phân tích cụm sau mô hình.

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
import torch
from sentence_transformers import SentenceTransformer
import umap

df = pd.read_csv(INPUT_PATH)

print("Kích thước dữ liệu ban đầu:", df.shape)

df_work = df.copy()

df_work["cluster_text_clean"] = (
    df_work["cluster_text_clean"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_work = df_work[df_work["cluster_text_clean"] != ""].copy()
df_work.reset_index(drop=True, inplace=True)

print("Kích thước dữ liệu sau khi bỏ văn bản rỗng:", df_work.shape)

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("\nThiết bị đang dùng:", device)
print("Đang tải model semantic:", MODEL_NAME)

model = SentenceTransformer(MODEL_NAME, device=device)

texts = df_work["cluster_text_clean"].tolist()

print("\nĐang tạo lại embedding semantic...")
X_semantic = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\nĐang giảm chiều bằng UMAP(50)...")
umap_reducer = umap.UMAP(
    n_neighbors=15,
    n_components=50,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)
X_semantic_umap = umap_reducer.fit_transform(X_semantic)

print("\nĐang gắn lại cụm semantic tốt nhất với K-Means(k=6)...")
kmeans_semantic = KMeans(
    n_clusters=6,
    init="k-means++",
    n_init=10,
    random_state=42
)

labels_semantic = kmeans_semantic.fit_predict(X_semantic_umap)
df_work["cluster_semantic"] = labels_semantic

print("\nSố lượng bài báo theo cụm:")
display(df_work["cluster_semantic"].value_counts().sort_index().to_frame("count"))

display(df_work[["doc_id", "category_clean", "cluster_text_lexical", "cluster_semantic"]].head(3))

Kích thước dữ liệu ban đầu: (8727, 9)
Kích thước dữ liệu sau khi bỏ văn bản rỗng: (8725, 9)

Thiết bị đang dùng: cuda
Đang tải model semantic: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...


Đang giảm chiều bằng UMAP(50)...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



Đang gắn lại cụm semantic tốt nhất với K-Means(k=6)...

Số lượng bài báo theo cụm:


,count
cluster_semantic,
0,891
1,1368
2,1976
3,1973
4,598
5,1919


,doc_id,category_clean,cluster_text_lexical,cluster_semantic
0,1,cong_nghe,làn_sóng máy_tính lượng_tử cơn_sốt ai quý thế_...,0
1,2,cong_nghe,meta đưa ai messenger trả_lời khách_hàng 24 7 ...,0
2,3,cong_nghe,hà_tiên sử_dụng trí_tuệ nhân_tạo phục_vụ hành_...,0


In [3]:
# Cell 4: rời rạc hóa và tạo transaction sạch cho FP-Max
# Cell này loại bỏ token nhiễu để chuẩn bị dữ liệu giao dịch cho khai thác mẫu theo từng cụm semantic.

import re

mining_stopwords = {
    'cho_biết', 'nói', 'rằng', 'theo', 'ngày', 'tháng', 'năm', 'đã', 'đang', 'sẽ',
    'được', 'việc', 'của', 'với', 'trong', 'tại', 'cũng', 'có_thể', 'cần', 'nhiều',
    'một_số', 'những', 'đối_với', 'liên_quan', 'thông_tin', 'tiếp_tục', 'yêu_cầu',
    'hiện_nay', 'thời_gian', 'cụ_thể', 'trực_tiếp', 'đặc_biệt', 'đồng_thời', 'vừa'
}

def clean_transaction(tokens):
    cleaned = []
    for t in tokens:
        t = str(t).lower().strip()
        if len(t) < 2:
            continue
        if t.isdigit():
            continue
        if t in mining_stopwords:
            continue
        if re.match(r'^[a-z_àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵ]+$', t):
            cleaned.append(t)
    return cleaned

cluster_transactions = {}

print("--- Đang chuẩn bị transaction sạch cho 6 cụm ---")
for cid in sorted(df_work["cluster_semantic"].unique()):
    raw_texts = df_work.loc[df_work["cluster_semantic"] == cid, "cluster_text_lexical"].astype(str)

    cleaned_trans = [clean_transaction(text.split()) for text in raw_texts]
    cleaned_trans = [tokens for tokens in cleaned_trans if len(tokens) > 0]

    cluster_transactions[cid] = cleaned_trans
    print(f"Cụm {cid}: {len(cleaned_trans)} văn bản")

print("\nHoàn tất chuẩn bị dữ liệu cho FP-Max.")
print("Ví dụ transaction đầu tiên của cụm 0:")
print(cluster_transactions[0][0][:20] if len(cluster_transactions[0]) > 0 else "Cụm 0 rỗng")

--- Đang chuẩn bị transaction sạch cho 6 cụm ---
Cụm 0: 891 văn bản
Cụm 1: 1368 văn bản
Cụm 2: 1976 văn bản
Cụm 3: 1973 văn bản
Cụm 4: 598 văn bản
Cụm 5: 1919 văn bản

Hoàn tất chuẩn bị dữ liệu cho FP-Max.
Ví dụ transaction đầu tiên của cụm 0:
['làn_sóng', 'máy_tính', 'lượng_tử', 'cơn_sốt', 'ai', 'quý', 'thế_giới', 'chứng_kiến', 'loạt', 'công_ty', 'vi_tính', 'lượng_tử', 'liên_tiếp', 'niêm_yết', 'sàn', 'chứng_khoán', 'mỹ', 'xem', 'tín_hiệu', 'rõ_rệt']


In [4]:
# Cell 5: chạy FP-Max cho từng cụm semantic
# Cell này tìm các mẫu phổ biến tối đại trong mỗi cụm để phục vụ khai thác mẫu và mô tả đặc trưng cụm.

try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import fpmax
    print("Đã import mlxtend thành công.")
except ImportError:
    !pip -q install mlxtend
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import fpmax
    print("Đã cài đặt và import mlxtend thành công.")

fpmax_results = {}

def choose_min_support(n_docs):
    if n_docs >= 1500:
        return 0.12
    elif n_docs >= 900:
        return 0.10
    else:
        return 0.08

print("--- Đang chạy FP-Max cho từng cụm ---")
for cid in sorted(cluster_transactions.keys()):
    transactions = cluster_transactions[cid]
    n_docs = len(transactions)
    min_support = choose_min_support(n_docs)

    te = TransactionEncoder()
    te_array = te.fit(transactions).transform(transactions)
    trans_df = pd.DataFrame(te_array, columns=te.columns_)

    patterns = fpmax(
        trans_df,
        min_support=min_support,
        use_colnames=True
    )

    if not patterns.empty:
        patterns["pattern_length"] = patterns["itemsets"].apply(len)
        patterns = patterns.sort_values(
            by=["support", "pattern_length"],
            ascending=[False, False]
        ).reset_index(drop=True)

    fpmax_results[cid] = patterns

    print(f"Cụm {cid}: {n_docs} văn bản | min_support = {min_support}")
    print(f"Số mẫu tối đại tìm được: {len(patterns)}")

print("\nHoàn tất chạy FP-Max cho 6 cụm.")

Đã import mlxtend thành công.
--- Đang chạy FP-Max cho từng cụm ---


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Cụm 0: 891 văn bản | min_support = 0.08
Số mẫu tối đại tìm được: 50983


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Cụm 1: 1368 văn bản | min_support = 0.1
Số mẫu tối đại tìm được: 5931


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Cụm 4: 598 văn bản | min_support = 0.08
Số mẫu tối đại tìm được: 11387


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Cụm 5: 1919 văn bản | min_support = 0.12
Số mẫu tối đại tìm được: 4932

Hoàn tất chạy FP-Max cho 6 cụm.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [5]:
# Cell 6: hiển thị và phân tích các mẫu FP-Max đặc trưng theo từng cụm
# Cell này lọc ra các mẫu tối đại có ý nghĩa hơn để hỗ trợ mô tả nội dung từng cụm.

print("--- KẾT QUẢ KHAI THÁC MẪU FP-MAX THEO CỤM ---")
print("-" * 70)

cluster_summary_data = []

for cid in sorted(fpmax_results.keys()):
    patterns = fpmax_results[cid]

    print(f"\n[PHÂN TÍCH CỤM {cid}]")

    if patterns.empty:
        print("Không tìm thấy mẫu nào thỏa mãn ngưỡng support.")
        cluster_summary_data.append({
            "Cluster": cid,
            "Top_Pattern": "",
            "Sample_Size": len(cluster_transactions[cid])
        })
        continue

    meaningful_patterns = patterns[patterns["pattern_length"] >= 2].copy()

    if meaningful_patterns.empty:
        meaningful_patterns = patterns.head(10).copy()
    else:
        meaningful_patterns = meaningful_patterns.head(10).copy()

    display(meaningful_patterns[["itemsets", "support", "pattern_length"]])

    top_pattern = meaningful_patterns.iloc[0]["itemsets"]
    top_pattern_sorted = ", ".join(sorted(list(top_pattern)))

    cluster_summary_data.append({
        "Cluster": cid,
        "Top_Pattern": top_pattern_sorted,
        "Sample_Size": len(cluster_transactions[cid])
    })

    print(f"Mẫu khung tiêu biểu: {top_pattern_sorted}")

print("\nHoàn tất trích xuất các mẫu đặc trưng.")

--- KẾT QUẢ KHAI THÁC MẪU FP-MAX THEO CỤM ---
----------------------------------------------------------------------

[PHÂN TÍCH CỤM 0]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,itemsets,support,pattern_length
0,"(người, sử_dụng, rủi_ro, dùng)",0.115600,4
1,"(sử_dụng, mới, nền_tảng, công_nghệ, người, dùng)",0.114478,6
2,"(cá_nhân, sử_dụng, mới, người, dùng)",0.114478,5
3,"(người, dùng, usd)",0.114478,3
4,"(sử_dụng, làm, mạng, người, dùng)",0.113356,5
5,"(hàng, người, mạng, dùng)",0.113356,4
6,"(người, sử_dụng, công_nghệ, tổ_chức)",0.113356,4
7,"(tin, mới, nhà, người)",0.113356,4
8,"(người, mới, bao_gồm)",0.113356,3
9,"(người, dùng, tỉ)",0.113356,3


Mẫu khung tiêu biểu: dùng, người, rủi_ro, sử_dụng

[PHÂN TÍCH CỤM 1]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

Mẫu khung tiêu biểu: người, thuộc, xe

[PHÂN TÍCH CỤM 5]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,itemsets,support,pattern_length
0,"(tp, trường, học, hcm, người)",0.173528,5
1,"(học, trường, giáo_dục, ông)",0.170922,4
2,"(người, trường, sinh_viên, học)",0.170401,4
3,"(học, trường, nam)",0.169880,3
4,"(cùng, trường, chương_trình, học)",0.168838,4
5,"(trường, tổ_chức, học, giáo_dục, học_sinh)",0.168317,5
6,"(giữ, trường)",0.168317,2
7,"(trường, ban)",0.167275,2
8,"(tp, trường, học, hcm, chương_trình)",0.166754,5
9,"(tốt_nghiệp, trường, học)",0.165711,3


Mẫu khung tiêu biểu: hcm, học, người, tp, trường

Hoàn tất trích xuất các mẫu đặc trưng.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [6]:
# Cell 8: lấy văn bản đại diện cho từng cụm semantic
# Cell này tìm các bài báo gần tâm cụm nhất trong không gian embedding để minh họa nội dung cụm.

from sklearn.metrics.pairwise import cosine_similarity

def get_representative_docs(df, embeddings, cluster_id, top_n=3):
    cluster_idx = df.index[df["cluster_semantic"] == cluster_id].to_numpy()
    cluster_embeds = embeddings[cluster_idx]

    centroid = cluster_embeds.mean(axis=0, keepdims=True)
    similarities = cosine_similarity(cluster_embeds, centroid).flatten()

    top_local_idx = similarities.argsort()[-top_n:][::-1]
    top_global_idx = cluster_idx[top_local_idx]

    rep_docs = df.loc[top_global_idx, ["doc_id", "category_clean", "cluster_text_clean"]].copy()
    rep_docs["similarity_to_centroid"] = similarities[top_local_idx]
    return rep_docs

print("--- VĂN BẢN ĐẠI DIỆN TIÊU BIỂU CHO MỖI CỤM ---")

representative_docs = {}

for cid in sorted(df_work["cluster_semantic"].unique()):
    print(f"\n[CỤM {cid}]")
    reps = get_representative_docs(df_work, X_semantic, cid, top_n=3)
    representative_docs[cid] = reps

    for i, (_, row) in enumerate(reps.iterrows(), start=1):
        preview = row["cluster_text_clean"][:220] + "..."
        print(
            f"{i}. doc_id={row['doc_id']} | gốc={row['category_clean']} "
            f"| sim={row['similarity_to_centroid']:.4f}"
        )
        print(f"   Nội dung: {preview}")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

--- VĂN BẢN ĐẠI DIỆN TIÊU BIỂU CHO MỖI CỤM ---

[CỤM 0]
1. doc_id=293 | gốc=cong_nghe | sim=0.7890
   Nội dung: ai thành cỗ máy lừa tự động : 75% thư rác, giả giọng ceo, bẫy tình trên mạng . ai đang đứng sau 50-75% thư rác, lừa đảo toàn cầu: nó quét mạng xã hội tìm nạn nhân yếu tâm lý, giả giọng ceo, dựng bẫy tinh vi chưa từng có....
2. doc_id=654 | gốc=cong_nghe | sim=0.7756
   Nội dung: người việt dùng internet gần 7 giờ mỗi ngày . người việt dùng internet với nhiều lĩnh vực, từ trao đổi thông tin, học tập, giao lưu kinh tế, giải trí...; đi kèm với phát triển là những thách thức về an ninh mạng ngày càn...
3. doc_id=960 | gốc=cong_nghe | sim=0.7534
   Nội dung: đầu năm 2025 đến nay phát hiện gần 1.500 vụ việc lừa đảo qua mạng . đánh giá về tình hình an ninh mạng việt nam đang phải đối mặt với nguy cơ và thách thức, đại diện cục a05 bộ công an đưa ra 3 vấn đề, trong đó tình trạn...

[CỤM 1]
1. doc_id=7808 | gốc=the_thao | sim=0.8493
   Nội dung: hlv vũ tiến thành: không nghĩ văn toản 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

### Nhận xét từ các văn bản đại diện

Các văn bản đại diện cho thấy mô hình semantic không chỉ tạo ra các cụm tốt về mặt chỉ số mà còn gom các bài báo có nội dung thực sự gần nhau về ngữ nghĩa. Cụm 1, Cụm 2 và Cụm 5 thể hiện rất rõ các chủ đề lần lượt là thể thao, du lịch và giáo dục. Cụm 0 nổi bật với các bài về công nghệ và an ninh mạng, trong khi Cụm 3 nghiêng về sức khỏe và y tế. Cụm 4 chủ yếu xoay quanh xe cộ và giao thông, dù vẫn có một số bài thuộc chuyên mục gốc khác nhưng nội dung thực tế vẫn gần với chủ đề giao thông.

Kết quả này cho thấy mô hình semantic đã gom cụm theo nội dung ngữ nghĩa thực sự, chứ không chỉ bám cơ học vào nhãn chuyên mục ban đầu. Đây là cơ sở quan trọng để tiếp tục sang bước kiểm định thống kê và so sánh độ ổn định của mô hình với baseline lexical.

In [7]:
# Cell 9: chuẩn bị hàm đánh giá cho kiểm định thống kê
# Cell này định nghĩa các metric dùng để so sánh baseline lexical và mô hình semantic trên nhiều lần chạy.

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)
from scipy.stats import wilcoxon

def purity_score(y_true, y_pred):
    contingency = pd.crosstab(pd.Series(y_true, name="true"), pd.Series(y_pred, name="pred"))
    return contingency.max(axis=0).sum() / contingency.values.sum()

def evaluate_clustering(y_true, X_space, labels):
    return {
        "Silhouette": silhouette_score(X_space, labels),
        "Davies_Bouldin": davies_bouldin_score(X_space, labels),
        "Calinski_Harabasz": calinski_harabasz_score(X_space, labels),
        "ARI": adjusted_rand_score(y_true, labels),
        "NMI": normalized_mutual_info_score(y_true, labels),
        "Purity": purity_score(y_true, labels)
    }

true_labels = df_work["category_clean"].values
print("Đã sẵn sàng các hàm đánh giá cho bước kiểm định thống kê.")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Đã sẵn sàng các hàm đánh giá cho bước kiểm định thống kê.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [8]:
# Cell 10: chạy thực nghiệm lặp (10 seeds) và thực hiện kiểm định Wilcoxon
# Cell này so sánh baseline lexical tốt nhất với mô hình semantic tốt nhất
# bằng kết quả thực nghiệm thật, không dùng dữ liệu giả lập.

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from scipy.stats import wilcoxon
import scipy.sparse as sp

class BM25Transformer:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b

    def fit(self, X):
        X = sp.csr_matrix(X)
        self.n_docs_ = X.shape[0]
        self.doc_freq_ = np.bincount(X.indices, minlength=X.shape[1])
        self.idf_ = np.log((self.n_docs_ - self.doc_freq_ + 0.5) / (self.doc_freq_ + 0.5) + 1.0)
        self.avgdl_ = X.sum(axis=1).A1.mean()
        return self

    def transform(self, X):
        X = sp.csr_matrix(X, dtype=np.float64)
        doc_lengths = X.sum(axis=1).A1
        len_norm = (1 - self.b) + self.b * (doc_lengths / self.avgdl_)

        data = X.data.copy()
        indices = X.indices
        indptr = X.indptr

        for i in range(X.shape[0]):
            start, end = indptr[i], indptr[i + 1]
            if start == end:
                continue

            norm_i = len_norm[i]
            term_idx = indices[start:end]
            tf = data[start:end]

            data[start:end] = self.idf_[term_idx] * (
                (tf * (self.k1 + 1)) / (tf + self.k1 * norm_i)
            )

        return sp.csr_matrix((data, indices, indptr), shape=X.shape)

print("--- Tái tạo baseline lexical tốt nhất: BM25 + SVD(300) ---")

count_vectorizer = CountVectorizer(
    min_df=5,
    max_df=0.85,
    max_features=10000
)

X_count = count_vectorizer.fit_transform(df_work["cluster_text_lexical"])

bm25_transformer = BM25Transformer(k1=1.5, b=0.75)
X_bm25 = bm25_transformer.fit(X_count).transform(X_count)

svd_lexical = TruncatedSVD(n_components=300, random_state=42)
X_lexical_svd = svd_lexical.fit_transform(X_bm25)

print("Kích thước X_lexical_svd:", X_lexical_svd.shape)
print("Kích thước X_semantic_umap:", X_semantic_umap.shape)

seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
lexical_runs = []
semantic_runs = []

print("\n--- ĐANG CHẠY THỰC NGHIỆM 10 LẦN ---")
for i, seed in enumerate(seeds, start=1):
    km_lex = KMeans(n_clusters=6, n_init=10, random_state=seed)
    labels_lex = km_lex.fit_predict(X_lexical_svd)
    lexical_runs.append(evaluate_clustering(true_labels, X_lexical_svd, labels_lex))

    km_sem = KMeans(n_clusters=6, n_init=10, random_state=seed)
    labels_sem = km_sem.fit_predict(X_semantic_umap)
    semantic_runs.append(evaluate_clustering(true_labels, X_semantic_umap, labels_sem))

    print(f"Hoàn tất vòng lặp {i}/10 với seed = {seed}")

df_lex_res = pd.DataFrame(lexical_runs)
df_sem_res = pd.DataFrame(semantic_runs)

results_summary = []

greater_metrics = ["ARI", "NMI", "Silhouette", "Purity", "Calinski_Harabasz"]
lower_metrics = ["Davies_Bouldin"]

for col in greater_metrics:
    stat, p_val = wilcoxon(df_sem_res[col], df_lex_res[col], alternative="greater")
    results_summary.append({
        "Metric": col,
        "Lexical (Mean ± Std)": f"{df_lex_res[col].mean():.4f} ± {df_lex_res[col].std():.4f}",
        "Semantic (Mean ± Std)": f"{df_sem_res[col].mean():.4f} ± {df_sem_res[col].std():.4f}",
        "p-value": round(p_val, 6),
        "Huong_ky_vong": "Semantic > Lexical",
        "Y_nghia_thong_ke": "CO" if p_val < 0.05 else "KHONG"
    })

for col in lower_metrics:
    stat, p_val = wilcoxon(df_sem_res[col], df_lex_res[col], alternative="less")
    results_summary.append({
        "Metric": col,
        "Lexical (Mean ± Std)": f"{df_lex_res[col].mean():.4f} ± {df_lex_res[col].std():.4f}",
        "Semantic (Mean ± Std)": f"{df_sem_res[col].mean():.4f} ± {df_sem_res[col].std():.4f}",
        "p-value": round(p_val, 6),
        "Huong_ky_vong": "Semantic < Lexical",
        "Y_nghia_thong_ke": "CO" if p_val < 0.05 else "KHONG"
    })

df_final_stat = pd.DataFrame(results_summary)
display(df_final_stat)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

--- Tái tạo baseline lexical tốt nhất: BM25 + SVD(300) ---


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Kích thước X_lexical_svd: (8725, 300)
Kích thước X_semantic_umap: (8725, 50)

--- ĐANG CHẠY THỰC NGHIỆM 10 LẦN ---


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Hoàn tất vòng lặp 10/10 với seed = 100


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Metric,Lexical (Mean ± Std),Semantic (Mean ± Std),p-value,Huong_ky_vong,Y_nghia_thong_ke
0,ARI,0.5390 ± 0.0032,0.7687 ± 0.0004,0.000977,Semantic > Lexical,CO
1,NMI,0.5926 ± 0.0026,0.7242 ± 0.0003,0.000977,Semantic > Lexical,CO
2,Silhouette,0.0272 ± 0.0001,0.6049 ± 0.0001,0.000977,Semantic > Lexical,CO
3,Purity,0.7683 ± 0.0021,0.8915 ± 0.0002,0.000977,Semantic > Lexical,CO
4,Calinski_Harabasz,208.6880 ± 0.0121,28421.0684 ± 0.0090,0.000977,Semantic > Lexical,CO
5,Davies_Bouldin,4.3285 ± 0.0065,0.5544 ± 0.0003,0.000977,Semantic < Lexical,CO


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

### Kết luận kiểm định thống kê

Kết quả kiểm định Wilcoxon signed-rank test cho thấy mô hình **Semantic Embedding + UMAP(50) + K-Means** vượt trội có ý nghĩa thống kê so với baseline **BM25 + SVD(300) + K-Means** trên toàn bộ các chỉ số được xét. Cụ thể, các chỉ số **ARI, NMI, Silhouette, Purity** và **Calinski-Harabasz** của mô hình semantic đều cao hơn có ý nghĩa thống kê, trong khi **Davies-Bouldin Index** thấp hơn có ý nghĩa thống kê. Tất cả các phép kiểm định đều cho `p-value = 0.000977 < 0.05`.

Ngoài ra, độ lệch chuẩn của các chỉ số ở mô hình semantic đều rất nhỏ qua 10 lần chạy với các seed khác nhau, cho thấy mô hình không chỉ cho kết quả tốt hơn mà còn có độ ổn định cao. Điều này củng cố mạnh mẽ kết luận rằng biểu diễn semantic là hướng tiếp cận phù hợp hơn rõ rệt so với biểu diễn lexical đối với bài toán gom cụm văn bản báo chí tiếng Việt trong bộ dữ liệu nghiên cứu này.

In [9]:
# Cell 12: tạo bảng tổng hợp cuối cho từng cụm semantic
# Cell này gom lại các thông tin quan trọng nhất của mỗi cụm để dễ chèn vào báo cáo.

cluster_summary_rows = []

for cid in sorted(df_work["cluster_semantic"].unique()):
    cluster_df = df_work[df_work["cluster_semantic"] == cid]

    dominant_category = cluster_df["category_clean"].value_counts().idxmax()
    dominant_count = cluster_df["category_clean"].value_counts().max()
    cluster_size = len(cluster_df)

    dominant_ratio = dominant_count / cluster_size

    top_pattern = ""
    if len(cluster_summary_data) > 0:
        matched = [x for x in cluster_summary_data if x["Cluster"] == cid]
        if len(matched) > 0:
            top_pattern = matched[0]["Top_Pattern"]

    rep_doc_id = None
    rep_similarity = None
    if cid in representative_docs:
        rep_doc = representative_docs[cid].iloc[0]
        rep_doc_id = rep_doc["doc_id"]
        rep_similarity = rep_doc["similarity_to_centroid"]

    cluster_summary_rows.append({
        "Cluster": cid,
        "Dominant_Category": dominant_category,
        "Cluster_Size": cluster_size,
        "Dominant_Ratio": round(dominant_ratio, 4),
        "Top_Pattern": top_pattern,
        "Representative_doc_id": rep_doc_id,
        "Representative_Similarity": round(rep_similarity, 4) if rep_similarity is not None else None
    })

cluster_summary_df = pd.DataFrame(cluster_summary_rows)
display(cluster_summary_df)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,Cluster,Dominant_Category,Cluster_Size,Dominant_Ratio,Top_Pattern,Representative_doc_id,Representative_Similarity
0,0,cong_nghe,891,0.8608,"dùng, người, rủi_ro, sử_dụng",293,0.7890
1,1,the_thao,1368,0.9437,"người, trận, vì",7808,0.8493
2,2,du_lich,1976,0.9013,"người, vì",2324,0.8565
3,3,suc_khoe,1973,0.9113,"người, nhận",6767,0.7816
4,4,xe,598,0.7274,"người, thuộc, xe",8711,0.7672
5,5,giao_duc,1919,0.8906,"hcm, học, người, tp, trường",3726,0.8483


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

### Kết luận cuối cùng của Notebook 5

Kết quả tổng hợp cho thấy mô hình semantic không chỉ vượt trội về mặt định lượng mà còn tạo ra các cụm có khả năng diễn giải tốt. Phần lớn các cụm đều có chuyên mục gốc chiếm ưu thế với tỉ lệ cao, đặc biệt ở các cụm thể thao, du lịch, sức khỏe và giáo dục. Các văn bản đại diện gần tâm cụm cũng phản ánh khá rõ nội dung chủ đạo của từng nhóm, cho thấy mô hình đã gom các bài báo gần nhau về ngữ nghĩa một cách hợp lý.

Bên cạnh đó, bước khai thác mẫu bằng FP-Max đã hỗ trợ nhận diện các từ khóa và mẫu từ xuất hiện phổ biến trong từng cụm. Tuy nhiên, một số mẫu vẫn còn chịu ảnh hưởng của các token quá chung, cho thấy phần khai thác mẫu có giá trị hỗ trợ diễn giải nhưng chưa sắc nét bằng biểu diễn semantic và các văn bản đại diện. Vì vậy, trong việc mô tả cụm, cần ưu tiên kết hợp giữa chuyên mục gốc chiếm ưu thế, văn bản đại diện và các mẫu FP-Max thay vì chỉ dựa vào riêng một nguồn thông tin.

Kết hợp với kết quả kiểm định thống kê ở phần trước, có thể kết luận rằng mô hình **Semantic Embedding + UMAP(50) + K-Means(k=6)** là cấu hình tốt nhất và ổn định nhất cho bài toán gom cụm văn bản báo chí tiếng Việt trong phạm vi dữ liệu của đồ án này.